In [18]:
import pandas as pd
from pathlib import Path

In [19]:
# ============================================================
# 1. PATH CONFIG FOR PROMPT 1
# ============================================================

BASE_DIR = Path("..")

PROMPT_ID = 1

PREPROCESSED_PATH = BASE_DIR / "dataset" / "asap" / "training_set_rel3_preprocessed.csv"
PROMPT_SCORE_PATH = BASE_DIR / "dataset" / "asap++" / "Prompt-1.csv"

OUTPUT_DIR = BASE_DIR / "dataset" / "asap++"
OUTPUT_PATH = OUTPUT_DIR / "Prompt-1-concat.csv"

print("Prompt ID:", PROMPT_ID)
print("Preprocessed path:", PREPROCESSED_PATH)
print("Prompt score path:", PROMPT_SCORE_PATH)
print("Output path:", OUTPUT_PATH)

Prompt ID: 1
Preprocessed path: ..\dataset\asap\training_set_rel3_preprocessed.csv
Prompt score path: ..\dataset\asap++\Prompt-1.csv
Output path: ..\dataset\asap++\Prompt-1-concat.csv


In [20]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df


def find_column(df: pd.DataFrame, candidates: list[str]) -> str:
    existing_cols = set(df.columns)

    for col in candidates:
        if col in existing_cols:
            return col

    raise ValueError(
        f"Không tìm thấy cột phù hợp.\n"
        f"Candidates: {candidates}\n"
        f"Existing columns: {list(df.columns)}"
    )


def preview_df(df: pd.DataFrame, name: str, n: int = 5):
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    display(df.head(n))

In [21]:
# ============================================================
# 3. LOAD DATA
# ============================================================

pre = pd.read_csv(PREPROCESSED_PATH)
scores = pd.read_csv(PROMPT_SCORE_PATH)

preview_df(pre, "Raw preprocessed")
preview_df(scores, f"Raw Prompt-{PROMPT_ID} scores")


===== Raw preprocessed =====
Shape: (12976, 5)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,1,1,"Dear local newspaper, I think effects computer...",8.0,0.6
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9.0,0.7
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7.0,0.5
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10.0,0.8
4,5,1,"Dear @LOCATION1, I know having computers has a...",8.0,0.6



===== Raw Prompt-1 scores =====
Shape: (1783, 6)
Columns: ['EssayID', 'Content', 'Organization', 'Word Choice', 'Sentence Fluency', 'Conventions']


,EssayID,Content,Organization,Word Choice,Sentence Fluency,Conventions
0,1,4,3,3,3,3
1,2,4,4,4,3,4
2,3,3,3,3,4,4
3,4,5,4,5,4,4
4,5,4,3,4,4,4


In [22]:
# ============================================================
# 4. NORMALIZE COLUMN NAMES
# ============================================================

pre = normalize_columns(pre)
scores = normalize_columns(scores)

preview_df(pre, "Normalized preprocessed")
preview_df(scores, f"Normalized Prompt-{PROMPT_ID} scores")


===== Normalized preprocessed =====
Shape: (12976, 5)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,1,1,"Dear local newspaper, I think effects computer...",8.0,0.6
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9.0,0.7
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7.0,0.5
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10.0,0.8
4,5,1,"Dear @LOCATION1, I know having computers has a...",8.0,0.6



===== Normalized Prompt-1 scores =====
Shape: (1783, 6)
Columns: ['essayid', 'content', 'organization', 'word_choice', 'sentence_fluency', 'conventions']


,essayid,content,organization,word_choice,sentence_fluency,conventions
0,1,4,3,3,3,3
1,2,4,4,4,3,4
2,3,3,3,3,4,4
3,4,5,4,5,4,4
4,5,4,3,4,4,4


In [23]:
# ============================================================
# 5. VALIDATE PREPROCESSED COLUMNS
# ============================================================

required_pre_cols = [
    "essay_id",
    "essay_set",
    "essay",
    "domain1_score",
    "domain1_score_norm"
]

missing_pre_cols = [col for col in required_pre_cols if col not in pre.columns]

if missing_pre_cols:
    raise ValueError(
        f"File preprocessed thiếu cột: {missing_pre_cols}\n"
        f"Các cột hiện có: {list(pre.columns)}"
    )

print("Preprocessed file OK.")

Preprocessed file OK.


In [24]:
# ============================================================
# 6. MAP SCORE COLUMNS
# ============================================================

essay_id_col = find_column(
    scores,
    [
        "essay_id",
        "essayid",
        "essay"
    ]
)

content_col = find_column(
    scores,
    [
        "content",
        "ideas_content",
        "ideas_and_content",
        "ideas__content",
        "ideas"
    ]
)

organization_col = find_column(
    scores,
    [
        "organization",
        "organizat",
        "organiza",
        "org"
    ]
)

word_choice_col = find_column(
    scores,
    [
        "word_choice",
        "word_cho",
        "wordchoice",
        "word"
    ]
)

sentence_fluency_col = find_column(
    scores,
    [
        "sentence_fluency",
        "sentence",
        "sent_fluency",
        "fluency"
    ]
)

conventions_col = find_column(
    scores,
    [
        "conventions",
        "convention",
        "conv"
    ]
)

print("Column mapping:")
print("essay_id:", essay_id_col)
print("content:", content_col)
print("organization:", organization_col)
print("word_choice:", word_choice_col)
print("sentence_fluency:", sentence_fluency_col)
print("conventions:", conventions_col)

Column mapping:
essay_id: essayid
content: content
organization: organization
word_choice: word_choice
sentence_fluency: sentence_fluency
conventions: conventions


In [25]:
# ============================================================
# 7. CLEAN PROMPT SCORE FILE
# ============================================================

prefix = f"p{PROMPT_ID}"

scores_clean = scores.rename(columns={
    essay_id_col: "essay_id",
    content_col: f"{prefix}_content",
    organization_col: f"{prefix}_organization",
    word_choice_col: f"{prefix}_word_choice",
    sentence_fluency_col: f"{prefix}_sentence_fluency",
    conventions_col: f"{prefix}_conventions"
})

score_cols = [
    f"{prefix}_content",
    f"{prefix}_organization",
    f"{prefix}_word_choice",
    f"{prefix}_sentence_fluency",
    f"{prefix}_conventions"
]

scores_clean = scores_clean[
    ["essay_id"] + score_cols
].copy()

preview_df(scores_clean, f"Clean Prompt-{PROMPT_ID} scores")


===== Clean Prompt-1 scores =====
Shape: (1783, 6)
Columns: ['essay_id', 'p1_content', 'p1_organization', 'p1_word_choice', 'p1_sentence_fluency', 'p1_conventions']


,essay_id,p1_content,p1_organization,p1_word_choice,p1_sentence_fluency,p1_conventions
0,1,4,3,3,3,3
1,2,4,4,4,3,4
2,3,3,3,3,4,4
3,4,5,4,5,4,4
4,5,4,3,4,4,4


In [26]:
# ============================================================
# 8. TYPE CONVERSION
# ============================================================

pre["essay_id"] = pd.to_numeric(pre["essay_id"], errors="coerce").astype("Int64")
pre["essay_set"] = pd.to_numeric(pre["essay_set"], errors="coerce").astype("Int64")
pre["domain1_score"] = pd.to_numeric(pre["domain1_score"], errors="coerce")
pre["domain1_score_norm"] = pd.to_numeric(pre["domain1_score_norm"], errors="coerce")

scores_clean["essay_id"] = pd.to_numeric(
    scores_clean["essay_id"],
    errors="coerce"
).astype("Int64")

for col in score_cols:
    scores_clean[col] = pd.to_numeric(scores_clean[col], errors="coerce")

preview_df(pre, "Preprocessed after type conversion")
preview_df(scores_clean, f"Prompt-{PROMPT_ID} scores after type conversion")


===== Preprocessed after type conversion =====
Shape: (12976, 5)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,1,1,"Dear local newspaper, I think effects computer...",8.0,0.6
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9.0,0.7
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7.0,0.5
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10.0,0.8
4,5,1,"Dear @LOCATION1, I know having computers has a...",8.0,0.6



===== Prompt-1 scores after type conversion =====
Shape: (1783, 6)
Columns: ['essay_id', 'p1_content', 'p1_organization', 'p1_word_choice', 'p1_sentence_fluency', 'p1_conventions']


,essay_id,p1_content,p1_organization,p1_word_choice,p1_sentence_fluency,p1_conventions
0,1,4,3,3,3,3
1,2,4,4,4,3,4
2,3,3,3,3,4,4
3,4,5,4,5,4,4
4,5,4,3,4,4,4


In [27]:
# ============================================================
# 9. FILTER PREPROCESSED TO CURRENT PROMPT ONLY
# ============================================================

pre_prompt = pre[pre["essay_set"] == PROMPT_ID].copy()

print(f"Rows in preprocessed Prompt {PROMPT_ID}:", len(pre_prompt))
print("Unique essay_set:", pre_prompt["essay_set"].unique())

preview_df(pre_prompt, f"Preprocessed Prompt {PROMPT_ID}")

Rows in preprocessed Prompt 1: 1783
Unique essay_set: <IntegerArray>
[1]
Length: 1, dtype: Int64

===== Preprocessed Prompt 1 =====
Shape: (1783, 5)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,1,1,"Dear local newspaper, I think effects computer...",8.0,0.6
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9.0,0.7
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7.0,0.5
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10.0,0.8
4,5,1,"Dear @LOCATION1, I know having computers has a...",8.0,0.6


In [28]:
# ============================================================
# 10. CHECK DUPLICATES BEFORE MERGE
# ============================================================

pre_dup_count = pre_prompt["essay_id"].duplicated().sum()
score_dup_count = scores_clean["essay_id"].duplicated().sum()

print("Duplicate essay_id in pre_prompt:", pre_dup_count)
print("Duplicate essay_id in scores_clean:", score_dup_count)

if pre_dup_count > 0:
    duplicated_ids = (
        pre_prompt
        .loc[pre_prompt["essay_id"].duplicated(), "essay_id"]
        .head(10)
        .tolist()
    )
    raise ValueError(
        f"Preprocessed Prompt {PROMPT_ID} có essay_id bị duplicate. "
        f"Ví dụ: {duplicated_ids}"
    )

if score_dup_count > 0:
    duplicated_ids = (
        scores_clean
        .loc[scores_clean["essay_id"].duplicated(), "essay_id"]
        .head(10)
        .tolist()
    )
    raise ValueError(
        f"Prompt-{PROMPT_ID}.csv có essay_id bị duplicate. "
        f"Ví dụ: {duplicated_ids}"
    )

Duplicate essay_id in pre_prompt: 0
Duplicate essay_id in scores_clean: 0


In [29]:
# ============================================================
# 11. MERGE HORIZONTAL BY essay_id
# ============================================================

concat_prompt = pre_prompt.merge(
    scores_clean,
    on="essay_id",
    how="inner"
)

print("Rows after merge:", len(concat_prompt))
preview_df(concat_prompt, f"Merged Prompt {PROMPT_ID}")

Rows after merge: 1783

===== Merged Prompt 1 =====
Shape: (1783, 10)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm', 'p1_content', 'p1_organization', 'p1_word_choice', 'p1_sentence_fluency', 'p1_conventions']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm,p1_content,p1_organization,p1_word_choice,p1_sentence_fluency,p1_conventions
0,1,1,"Dear local newspaper, I think effects computer...",8.0,0.6,4,3,3,3,3
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9.0,0.7,4,4,4,3,4
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7.0,0.5,3,3,3,4,4
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10.0,0.8,5,4,5,4,4
4,5,1,"Dear @LOCATION1, I know having computers has a...",8.0,0.6,4,3,4,4,4


In [30]:
# ============================================================
# 12. CHECK MISSING IDS
# ============================================================

pre_ids = set(pre_prompt["essay_id"].dropna().astype(int))
score_ids = set(scores_clean["essay_id"].dropna().astype(int))
merged_ids = set(concat_prompt["essay_id"].dropna().astype(int))

missing_in_scores = sorted(list(pre_ids - score_ids))
missing_in_pre = sorted(list(score_ids - pre_ids))

print(f"Prompt {PROMPT_ID} essays in preprocessed:", len(pre_ids))
print(f"Prompt {PROMPT_ID} essays in score file:", len(score_ids))
print("Merged essays:", len(merged_ids))

print("IDs có trong preprocessed nhưng không có trong score file:", len(missing_in_scores))
print("IDs có trong score file nhưng không có trong preprocessed prompt:", len(missing_in_pre))

if len(missing_in_scores) > 0:
    print("Ví dụ missing_in_scores:", missing_in_scores[:20])

if len(missing_in_pre) > 0:
    print("Ví dụ missing_in_pre:", missing_in_pre[:20])

Prompt 1 essays in preprocessed: 1783
Prompt 1 essays in score file: 1783
Merged essays: 1783
IDs có trong preprocessed nhưng không có trong score file: 0
IDs có trong score file nhưng không có trong preprocessed prompt: 0


In [31]:
# ============================================================
# 13. DROP NaN TRAIT SCORES
# ============================================================

before_drop = len(concat_prompt)

concat_prompt = concat_prompt.dropna(subset=score_cols).copy()

after_drop = len(concat_prompt)

print("Rows before drop NaN trait scores:", before_drop)
print("Rows after drop NaN trait scores:", after_drop)
print("Dropped rows:", before_drop - after_drop)

Rows before drop NaN trait scores: 1783
Rows after drop NaN trait scores: 1783
Dropped rows: 0


In [32]:
# ============================================================
# 14. VALIDATE PROMPT 1 TRAIT SCORE RANGE
# ============================================================

# Prompt 1 trait scores theo guideline là 1-6
for col in score_cols:
    invalid_mask = ~concat_prompt[col].between(1, 6)
    invalid_count = invalid_mask.sum()

    print(f"{col} invalid count:", invalid_count)

    if invalid_count > 0:
        display(concat_prompt.loc[invalid_mask, ["essay_id", col]].head(20))

p1_content invalid count: 0
p1_organization invalid count: 0
p1_word_choice invalid count: 0
p1_sentence_fluency invalid count: 0
p1_conventions invalid count: 0


In [33]:
# ============================================================
# 15. ADD prompt_id AND REORDER COLUMNS
# ============================================================

concat_prompt["prompt_id"] = PROMPT_ID

final_cols = [
    "essay_id",
    "essay_set",
    "prompt_id",
    "essay",
    "domain1_score",
    "domain1_score_norm",
] + score_cols

concat_prompt = concat_prompt[final_cols].copy()

preview_df(concat_prompt, f"Final Prompt-{PROMPT_ID} concat")


===== Final Prompt-1 concat =====
Shape: (1783, 11)
Columns: ['essay_id', 'essay_set', 'prompt_id', 'essay', 'domain1_score', 'domain1_score_norm', 'p1_content', 'p1_organization', 'p1_word_choice', 'p1_sentence_fluency', 'p1_conventions']


,essay_id,essay_set,prompt_id,essay,domain1_score,domain1_score_norm,p1_content,p1_organization,p1_word_choice,p1_sentence_fluency,p1_conventions
0,1,1,1,"Dear local newspaper, I think effects computer...",8.0,0.6,4,3,3,3,3
1,2,1,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9.0,0.7,4,4,4,3,4
2,3,1,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7.0,0.5,3,3,3,4,4
3,4,1,1,"Dear Local Newspaper, @CAPS1 I have found that...",10.0,0.8,5,4,5,4,4
4,5,1,1,"Dear @LOCATION1, I know having computers has a...",8.0,0.6,4,3,4,4,4


In [34]:
# ============================================================
# 16. SAVE OUTPUT
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

concat_prompt.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)
print("Final shape:", concat_prompt.shape)

Saved to: ..\dataset\asap++\Prompt-1-concat.csv
Final shape: (1783, 11)


In [35]:
# ============================================================
# 17. QUICK SANITY CHECK
# ============================================================

print("Essay set counts:")
print(concat_prompt["essay_set"].value_counts(dropna=False))

print("\nPrompt id counts:")
print(concat_prompt["prompt_id"].value_counts(dropna=False))

print("\nTrait score describe:")
display(concat_prompt[score_cols].describe())

print("\nNaN count:")
print(concat_prompt.isna().sum())

Essay set counts:
essay_set
1    1783
Name: count, dtype: Int64

Prompt id counts:
prompt_id
1    1783
Name: count, dtype: int64

Trait score describe:


,p1_content,p1_organization,p1_word_choice,p1_sentence_fluency,p1_conventions
count,1783.000000,1783.000000,1783.000000,1783.000000,1783.000000
mean,3.846887,3.737521,3.679192,3.762759,3.737521
std,0.991320,0.950858,0.966469,0.968832,0.948494
min,1.000000,1.000000,1.000000,1.000000,1.000000
25%,3.000000,3.000000,3.000000,3.000000,3.000000
50%,4.000000,4.000000,4.000000,4.000000,4.000000
75%,4.000000,4.000000,4.000000,4.000000,4.000000
max,6.000000,6.000000,6.000000,6.000000,6.000000



NaN count:
essay_id               0
essay_set              0
prompt_id              0
essay                  0
domain1_score          0
domain1_score_norm     0
p1_content             0
p1_organization        0
p1_word_choice         0
p1_sentence_fluency    0
p1_conventions         0
dtype: int64
